# ChunkRAG main study — canonical Colab shard
Implements Immutable Specification Section 28. Run only on an A100 runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
PROTOCOL_SHA256 = '567b652fc403e7ff7e00e349de86357f9a293cac77e7e7f4d3612284eb2c89bf'
GIT_COMMIT = 'SET_ME'
EXPERIMENT = 'E2'
MODEL = 'mistral'
DATASET = 'squad_v2'
CONDITION_ID = 'recursive192__matched-4096'
SHARD_INDEX = 0
QUESTION_MANIFEST_HASH = 'SET_ME'
UPSTREAM_HASH = 'SET_ME'


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
gpu = subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], text=True).strip()
assert 'A100' in gpu, f'Noncanonical GPU: {gpu}'
assert GIT_COMMIT != 'SET_ME' and QUESTION_MANIFEST_HASH != 'SET_ME' and UPSTREAM_HASH != 'SET_ME'
REPO = '/content/chunkrag-course-project'
subprocess.run(['git','-C',REPO,'checkout',GIT_COMMIT], check=True)
subprocess.run(['git','-C',REPO,'status','--porcelain'], check=True, capture_output=True, text=True).stdout == '' or (_ for _ in ()).throw(RuntimeError('dirty checkout'))
subprocess.run(['python','-m','pip','install','-r',f'{REPO}/requirements-main-study.lock'], check=True)
sys.path.insert(0, f'{REPO}/src')
from chunkrag.mainstudy.environment import environment_manifest
lock = environment_manifest(Path(REPO) / 'requirements-main-study.lock', check_installed=True)
runtime = {'protocol_sha256': PROTOCOL_SHA256, 'git_commit': GIT_COMMIT, 'environment_hash': lock['lock_sha256'], 'hardware': lock['hardware']}
runtime_path = '/content/chunkrag-runtime.json'
Path(runtime_path).write_text(json.dumps(runtime, sort_keys=True) + '\n')
artifact_root = f'/content/drive/MyDrive/chunkrag-main-v1/{GIT_COMMIT}'
completed = {'E2':['E1'],'E3':['E0'],'E4':['E2','E3'],'E5':['E1'],'E6':['E1','E3'],'E7':['E0','E1','E2','E3','E4','E5','E6']}.get(EXPERIMENT, [])
command = ['python',f'{REPO}/scripts/run_main_study.py','--experiment',EXPERIMENT,'--mode','run','--platform','colab','--runtime-manifest',runtime_path,'--artifact-root',artifact_root,'--dataset',DATASET,'--condition-id',CONDITION_ID,'--shard-index',str(SHARD_INDEX),'--completed',*completed]
subprocess.run(command, check=True, env={**os.environ, 'PYTHONPATH': f'{REPO}/src'})
